# S11 — Transfer Learning and Data Augmentation

**Week 6 · Wed Sep 30, 2026 · Module 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s11_transfer_learning_and_data_augmentation.ipynb)

Every cell below is a worked example from the [S11 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s11/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s11.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s11.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
print("environment ready")

## Transfer in miniature: a complete, self-contained experiment


*Expected output starts with:* `seed 0: transfer = 0.7640, scratch = 0.5940`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

# Task A (pretraining): classify the orientation of a noisy bar (4 classes).
# Task B (target):      classify "+" vs "x" -- compositions of those bars.
DIRS = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (1, -1)}  # 0, 45, 90, 135 degrees

def draw_bar(img, r, c, d):
    dr, dc = DIRS[d]
    for t in range(-3, 4):
        rr, cc = r + t * dr, c + t * dc
        if 0 <= rr < 16 and 0 <= cc < 16:
            img[rr, cc] = 1.0

def make_task_a(n, rng):
    X = rng.normal(0.0, 0.4, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 4, size=n)
    for i in range(n):
        r, c = rng.integers(4, 12, size=2)
        draw_bar(X[i], r, c, int(y[i]))
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_task_b(n, rng):
    X = rng.normal(0.0, 0.4, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 2, size=n)
    for i in range(n):
        r, c = rng.integers(4, 12, size=2)
        if y[i] == 0:
            draw_bar(X[i], r, c, 0); draw_bar(X[i], r, c, 2)   # "+"
        else:
            draw_bar(X[i], r, c, 1); draw_bar(X[i], r, c, 3)   # "x"
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_trunk():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),                       # -> 16 * 4 * 4 = 256 features
    )

def train(model, X, y, epochs, params=None):
    params = list(model.parameters()) if params is None else list(params)
    opt = torch.optim.Adam(params, lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            loss_fn(model(X[idx]), y[idx]).backward()
            opt.step()

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

results = {"transfer": [], "scratch": []}
for seed in range(3):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    Xa, ya = make_task_a(2000, rng)             # task A: plenty of data
    Xb, yb = make_task_b(24, rng)               # task B: only 24 examples
    Xb_te, yb_te = make_task_b(500, rng)

    # 1) Pretrain trunk + 4-way head on task A.
    pre = nn.Sequential(make_trunk(), nn.Linear(256, 4))
    train(pre, Xa, ya, epochs=8)

    # 2) Transfer: keep the trunk, freeze it, train a fresh 2-way head.
    trunk = pre[0]
    for p in trunk.parameters():
        p.requires_grad = False
    torch.manual_seed(seed + 100)
    head = nn.Linear(256, 2)
    transfer = nn.Sequential(trunk, head)
    train(transfer, Xb, yb, epochs=150, params=head.parameters())

    # 3) Baseline: identical architecture, trained from scratch on task B.
    torch.manual_seed(seed + 100)
    scratch = nn.Sequential(make_trunk(), nn.Linear(256, 2))
    train(scratch, Xb, yb, epochs=150)

    results["transfer"].append(accuracy(transfer, Xb_te, yb_te))
    results["scratch"].append(accuracy(scratch, Xb_te, yb_te))
    print(f"seed {seed}: transfer = {results['transfer'][-1]:.4f}, "
          f"scratch = {results['scratch'][-1]:.4f}")

for name, accs in results.items():
    print(f"{name:8s} mean test accuracy = {np.mean(accs):.4f}")

## Probe, fine-tune, or both: the choice, measured


*Expected output starts with:* `--- 24 target examples ---`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)   # small model: thread overhead outweighs parallelism

# Linear probe vs full fine-tuning, at two target-data sizes.
# Same tasks as the transfer experiment: task A = bar orientation (source),
# task B = "+" vs "x" (target).
DIRS = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (1, -1)}

def draw_bar(img, r, c, d):
    dr, dc = DIRS[d]
    for t in range(-3, 4):
        rr, cc = r + t * dr, c + t * dc
        if 0 <= rr < 16 and 0 <= cc < 16:
            img[rr, cc] = 1.0

def make_task_a(n, rng):
    X = rng.normal(0.0, 0.4, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 4, size=n)
    for i in range(n):
        r, c = rng.integers(4, 12, size=2)
        draw_bar(X[i], r, c, int(y[i]))
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_task_b(n, rng):
    X = rng.normal(0.0, 0.4, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 2, size=n)
    for i in range(n):
        r, c = rng.integers(4, 12, size=2)
        if y[i] == 0:
            draw_bar(X[i], r, c, 0); draw_bar(X[i], r, c, 2)
        else:
            draw_bar(X[i], r, c, 1); draw_bar(X[i], r, c, 3)
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_trunk():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
    )

def train(model, X, y, epochs, param_groups):
    opt = torch.optim.Adam(param_groups)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            loss_fn(model(X[idx]), y[idx]).backward()
            opt.step()

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

conds = ["probe (frozen trunk)", "fine-tune (trunk lr/10)", "fine-tune (one big lr)"]
results = {(c, n): [] for c in conds for n in [24, 200]}
for seed in range(3):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    Xa, ya = make_task_a(2000, rng)
    pre = nn.Sequential(make_trunk(), nn.Linear(256, 4))
    train(pre, Xa, ya, epochs=8, param_groups=[{"params": pre.parameters(), "lr": 1e-3}])
    state = {k: v.clone() for k, v in pre[0].state_dict().items()}

    for n_target in [24, 200]:
        Xb, yb = make_task_b(n_target, rng)
        Xb_te, yb_te = make_task_b(500, rng)
        for cond in conds:
            trunk = make_trunk()
            trunk.load_state_dict(state)      # same pretrained weights each time
            torch.manual_seed(seed + 100)
            head = nn.Linear(256, 2)
            model = nn.Sequential(trunk, head)
            if cond.startswith("probe"):
                for p in trunk.parameters():
                    p.requires_grad = False
                groups = [{"params": head.parameters(), "lr": 1e-3}]
            elif "lr/10" in cond:             # discriminative learning rates
                groups = [{"params": head.parameters(), "lr": 1e-3},
                          {"params": trunk.parameters(), "lr": 1e-4}]
            else:                             # one aggressive lr for everything
                groups = [{"params": model.parameters(), "lr": 1e-2}]
            train(model, Xb, yb, epochs=150, param_groups=groups)
            results[(cond, n_target)].append(accuracy(model, Xb_te, yb_te))

for n_target in [24, 200]:
    print(f"--- {n_target} target examples ---")
    for cond in conds:
        accs = results[(cond, n_target)]
        print(f"{cond:24s} mean = {np.mean(accs):.4f} "
              f"(seeds: {', '.join(f'{a:.3f}' for a in accs)})")

## Augmentation that helps, augmentation that lies


*Expected output starts with:* `no augmentation                       test acc = 0.7147 (seeds: 0.706, 0.634, 0.804)`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

def draw_L(img, r, c, mirror=False):
    """An L shape (or its mirror image) with top corner at (r, c)."""
    col = c + 4 if mirror else c
    for t in range(7):
        img[r + t, col] = 1.0            # vertical stroke
    for t in range(5):
        img[r + 6, c + t] = 1.0          # horizontal foot
    return img

def make_data(n, rng):
    """Classify L (class 0) vs mirrored L (class 1) at random positions."""
    X = rng.normal(0.0, 0.15, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 2, size=n)
    for i in range(n):
        r = rng.integers(0, 9)
        c = rng.integers(0, 11)
        draw_L(X[i], r, c, mirror=bool(y[i]))
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_model():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(256, 2),
    )

def shift_aug(X, rng):
    """Random translation up to 3 px: label-PRESERVING for this task."""
    out = torch.empty_like(X)
    for i in range(len(X)):
        dr, dc = rng.integers(-3, 4, size=2)
        out[i] = torch.roll(X[i], shifts=(int(dr), int(dc)), dims=(1, 2))
    return out

def flip_aug(X, rng):
    """Random horizontal flip: label-DESTROYING for this task."""
    out = X.clone()
    mask = torch.from_numpy(rng.random(len(X)) < 0.5)
    out[mask] = torch.flip(out[mask], dims=[3])
    return out

def run(augment, seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    Xtr, ytr = make_data(32, rng)            # tiny training set
    Xte, yte = make_data(500, rng)
    model = make_model()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(400):
        Xb = augment(Xtr, rng) if augment else Xtr
        opt.zero_grad()
        loss_fn(model(Xb), ytr).backward()
        opt.step()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

for name, aug in [("no augmentation", None),
                  ("random shifts (label-preserving)", shift_aug),
                  ("horizontal flips (label-destroying)", flip_aug)]:
    accs = [run(aug, seed) for seed in range(3)]
    print(f"{name:37s} test acc = {np.mean(accs):.4f} "
          f"(seeds: {', '.join(f'{a:.3f}' for a in accs)})")

## When transfer fails


*Expected output starts with:* `matched domain             frozen transfer = 0.8287, scratch = 0.6707`


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)   # small model: thread overhead outweighs parallelism

# When does transfer fail? Pretrain on bright bars (task A), then transfer to
# a target task whose domain either matches (bright "+" vs "x") or does not
# (contrast-INVERTED "+" vs "x": dark shapes on bright noise).
DIRS = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (1, -1)}

def draw_bar(img, r, c, d):
    dr, dc = DIRS[d]
    for t in range(-3, 4):
        rr, cc = r + t * dr, c + t * dc
        if 0 <= rr < 16 and 0 <= cc < 16:
            img[rr, cc] = 1.0

def make_task_a(n, rng):
    X = rng.normal(0.0, 0.4, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 4, size=n)
    for i in range(n):
        r, c = rng.integers(4, 12, size=2)
        draw_bar(X[i], r, c, int(y[i]))
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_task_b(n, rng, invert):
    X = rng.normal(0.0, 0.4, size=(n, 16, 16)).astype(np.float32)
    y = rng.integers(0, 2, size=n)
    for i in range(n):
        r, c = rng.integers(4, 12, size=2)
        if y[i] == 0:
            draw_bar(X[i], r, c, 0); draw_bar(X[i], r, c, 2)
        else:
            draw_bar(X[i], r, c, 1); draw_bar(X[i], r, c, 3)
    if invert:
        X = -X                     # same shapes, opposite contrast polarity
    return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(y).long()

def make_trunk():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
    )

def train(model, X, y, epochs, params=None):
    params = list(model.parameters()) if params is None else list(params)
    opt = torch.optim.Adam(params, lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            idx = perm[i:i + 64]
            opt.zero_grad()
            loss_fn(model(X[idx]), y[idx]).backward()
            opt.step()

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

for invert in [False, True]:
    tag = "shifted domain (inverted)" if invert else "matched domain"
    frozen_accs, scratch_accs = [], []
    for seed in range(3):
        rng = np.random.default_rng(seed)
        torch.manual_seed(seed)
        Xa, ya = make_task_a(2000, rng)
        Xb, yb = make_task_b(24, rng, invert)
        Xb_te, yb_te = make_task_b(500, rng, invert)

        pre = nn.Sequential(make_trunk(), nn.Linear(256, 4))
        train(pre, Xa, ya, epochs=8)
        trunk = pre[0]
        for p in trunk.parameters():
            p.requires_grad = False
        torch.manual_seed(seed + 100)
        head = nn.Linear(256, 2)
        transfer = nn.Sequential(trunk, head)
        train(transfer, Xb, yb, epochs=150, params=head.parameters())
        frozen_accs.append(accuracy(transfer, Xb_te, yb_te))

        torch.manual_seed(seed + 100)
        scratch = nn.Sequential(make_trunk(), nn.Linear(256, 2))
        train(scratch, Xb, yb, epochs=150)
        scratch_accs.append(accuracy(scratch, Xb_te, yb_te))

    print(f"{tag:26s} frozen transfer = {np.mean(frozen_accs):.4f}, "
          f"scratch = {np.mean(scratch_accs):.4f}")

## Try it yourself

1. Add a third condition to the transfer experiment: *full fine-tuning* — initialize from the pretrained trunk but leave every parameter trainable on task B's 24 examples. Where does it land relative to frozen and scratch, and does the answer change with 200 task-B examples?
2. Make tasks A and B less related: pretrain on bars that only ever appear in the top-left quadrant, or on horizontal/vertical bars only (dropping the diagonals), and rerun. How much of the transfer advantage survives?
3. In the augmentation experiment, add vertical flips and 90-degree rotations. Predict, before running, whether each is label-preserving for the L task; then check.
4. Combine both: transfer the task-A trunk to the L-vs-mirror-L task with shift augmentation. Do the two techniques stack, or does one subsume the other here?


---

Full discussion of everything above: [S11 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s11/).
